In [1]:
import os
import sys
import multiprocessing


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyProBound_operator as pbo
import logomaker

In [2]:
print('#######################################################################################')
print('#### Data Wrangling with SmileSeq sequences for revisions ©Antoni Gralak_23.05.2025####')
print('#######################################################################################')

#######################################################################################
#### Data Wrangling with SmileSeq sequences for revisions ©Antoni Gralak_23.05.2025####
#######################################################################################


In [3]:
print('Setting env...')

this_path = os.getcwd()
sys.path.append(os.path.join(this_path, '..'))

num_cores = 2

experiment_name = 'exp10'
                #Chooses from:
                #['exp1', 'exp2', 'exp3', 'exp4', 'exp5',
                #'exp6', 'exp7', 'exp8', 'exp9', 'exp10',
                #'exp11', 'exp12', 'exp13', 'exp14', 'exp15',
                #'exp16', 'exp17', 'exp18', 'exp19', 'exp20',
                #'exp21', 'exp22', 'exp23']

to_be_analyzed = ['ZNF891_FL']

Setting env...


In [4]:
# Load metadata
metadata = pd.read_csv('../metadata.csv', sep=',')

data_path = '../output_joint_analysis/00_read_in_data/'

save_path = this_path + '/output_separate_analysis/'
os.makedirs(save_path, exist_ok=True)

In [5]:
# Since I some barcodes might have mutations, these two functions will allow to identify the mBCs with
# a Hamming distance 2 and chage them to the respective BC.


def _hamming_distance(s1, s2):
    """
    Calculates the Hamming distance between two strings s1 and s2.
    """
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def _find_similar_strings(input_str, strings):
    """
    Finds strings in the list `strings` that have a Hamming distance of 1
    from the input string `input_str`.
    """
    similar_strings = []
    for s in strings:
        if _hamming_distance(input_str, s) < 2:
            similar_strings.append(True)
        else:
            similar_strings.append(False)
    return similar_strings


def barcode_correction(df, methylation_BC):
    """
    Corrects barcodes in the 'methl' column of the DataFrame by replacing 
    any string within Hamming distance <= 1 of the reference barcodes.

    Parameters:
    - df: pandas DataFrame with a 'methl' column
    - methylated_BC: reference string for methylated barcode (e.g. 'AGTA')
    - unmethylated_BC: reference string for unmethylated barcode (e.g. 'GAAT')

    Returns:
    - df: corrected DataFrame (modified in place)
    """
    mBCs = df['methl']

    similar_to_meth = _find_similar_strings(methylation_BC, mBCs)
    df.loc[similar_to_meth, 'methl'] = methylation_BC

    return df

In [6]:
if to_be_analyzed is not None:
    curated_data = to_be_analyzed
else:
    curated_data = list(metadata[metadata['approved'] == True]['TF'])

In [7]:
# read in data of choice and perform motif discovery separately
all_dfs = []
for TF in curated_data:

    print(f'loading necessary datasets... {TF}')
    

    if experiment_name in ['exp1', 'exp2', 'exp3']:
        input_id = 'input1' 
        methylated_BC = "AGTA"
        unmethylated_BC = "GAGT"
        #flanking regions of the library and corresponding parameter for ProBound model
        left = ''
        right = ''
        binding_mode_flank=5
    elif experiment_name in ['exp4', 'exp5', 'exp6', 'exp7', 'exp8']:
        input_id = 'input2'
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        left = 'GGGGTACTGTGGAGATAG'
        right = 'AAACTCCCTGAGACC'
        binding_mode_flank=18
    elif experiment_name in ['exp9', 'exp10', 'exp11', 'exp12', 'exp13', 'exp14']:
        input_id = 'input3'
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        left = 'GGGGTACTGTGGAGATAG'
        right = 'AAACTCCCTGAGACC'
        binding_mode_flank=18
    elif experiment_name in ['exp15', 'exp16', 'exp17', 'exp18', 'exp19', 'exp20', 'exp21', 'exp22', 'exp23']:
        input_id = 'input4'
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        left = 'GGGGTACTGTGGAGATAG'
        right = 'AAACTCCCTGAGACC'
        binding_mode_flank=18
    else:
        print("Experiment_name needs to be a experiment ID, e.g. exp1 (possible options 1 to 23). Stopping script.")
        sys.exit(1)

    position_on_chip = metadata[(metadata['experiment'] == experiment_name) & (metadata['TF'] == TF)]['Chip_pos'].values[0]
    input_path = data_path + f'{input_id}_{position_on_chip}_raw_data.csv'
    eluted_df = pd.read_csv(data_path + f'{experiment_name}_{TF}_raw_data.csv')
    

    eluted_df = barcode_correction(eluted_df, methylated_BC)
    eluted_df = barcode_correction(eluted_df, unmethylated_BC)

    #split based on methylated BC, for separate analysis
    methl_df = eluted_df[eluted_df['methl'] == methylated_BC].reset_index(drop=True)
    unmethl_df = eluted_df[eluted_df['methl'] == unmethylated_BC].reset_index(drop=True)


    all_dfs.append(('methylated', experiment_name, position_on_chip, TF, input_path, methl_df))
    all_dfs.append(('unmethylated', experiment_name, position_on_chip, TF, input_path, unmethl_df))

loading necessary datasets... ZNF891_FL


In [ ]:
# Use split data by methylation to infer separate motifs.

def process_files(chunk):
    for mstat, exp, pos, TF, input, df in chunk:
    
        # output_paths
        results_path = save_path + f'{experiment_name}/results/{TF}/'
        misc_path = save_path + f'{experiment_name}/misc/{TF}/'

        
        ######################################################
        
        os.makedirs(results_path, exist_ok=True)
        
        os.makedirs(misc_path, exist_ok=True)
    
        ######################################################

        #load correct input files
       
        input_df = pd.read_csv(input)

        input_df = barcode_correction(input_df, df['methl'][0])


        input_df = input_df[input_df['methl'] == df['methl'][0]].reset_index(drop=True)

        # Defining variables for ProBound
        #left flank
        left_flank = df[['methl', 'BC', 'lfl']].apply(lambda row: ''.join(row.values.astype(str)), axis=1)[0]
        right_flank = df['rfl'][0]
        binding_mode_flank = 5

        binding_mode_size = [6] #[6,9,12,15] 

        #creating a df for pyProBound
        input_PB = pd.DataFrame(
                                {'header': np.repeat('input', len(input_df))
                                })
        input_PB['sequence'] = list(input_df['random24'])

            

        eluted_PB = pd.DataFrame(
                                {'header': np.repeat('eluted', len(df))
                                })
        eluted_PB['sequence'] = list(df['random24'])


        # run ProBound
        for binding_mode in binding_mode_size:

            #########################################
            # Create config for ProBound and set env#
            #########################################
            print('Create config for ProBound and set env')
            outputfile = misc_path + f'f_{TF}_bm{binding_mode}_{mstat}_output.tsv'
            count_table = pbo.build_count_table(input_PB, eluted_PB,
                                        output_filename=outputfile, gzip=False)
            
            # the default tested configuration for smile seq with three binding modes
            config = pbo.generate_SMiLE_seq_configuration(outputfile,
                                                        variable_region_length=24,
                                                        left_flank=left_flank,
                                                        right_flank=right_flank,
                                                        binding_mode_flank=binding_mode_flank, 
                                                          # this must be smaller than the left and right flank size
                                                        binding_modes=3,
                                                        binding_mode_size=binding_mode)
            basename = TF + f'_bm{binding_mode}_testmodel_{mstat}'

            config.alter_output(output_path=misc_path, 
                                base_name=basename, 
                                print_trajectory=True, 
                                # if true, generates several files in the output path, one set for each binding mode
                                # <base_name>.trajectory.component<binding mode index>-<desc of file>.csv
                                verbose=False) # flipping this switch does not seem to do very much tbh
            
            # Once you are done with the config modifications, write it to file. 
            # For ProBound, only what is written to the config file counts!
            
            config_filename = misc_path + f'{TF}_bm{binding_mode}_{mstat}_config.json'
            config.print_json(config_filename)
            
            # Run ProBound
            print(f'Running ProBound for {TF}, {exp}, binding size {binding_mode}, {mstat}.')
            pbo.run_probound(config_filename, 
                             full_config_file=misc_path + f"{mstat}_tmp.fullconfig.json",
                             save_output=misc_path + f"{mstat}_tmp.optimization.out",
                             cleanup_verbose=True)
            
            # Retrieve psam
            result_filename = pbo.get_psam(misc_path + f"{basename}.models.json")
            
            # Extract psam and plot dGG matrix
            for j, psam in enumerate(result_filename):
                psam.to_csv(results_path + f'{TF}_bm{binding_mode}_{mstat}_bindingmode_{str(j + 1)}.csv')

                fig, ax = plt.subplots(1,1,figsize=[10,6])
                logo = logomaker.Logo(result_filename[j],
                                    shade_below=0.5,
                                    ax=ax,
                                    fade_below=0.5,
                                    color_scheme={'A':'#66a61e', 'C':'#7570b3','G':'#ffc809','T':'#d95f02','m':'#a6cee3'}
                                    )
                # style using Logo methods
                logo.style_spines(visible=False)
                logo.style_spines(spines=['left', 'bottom'], visible=True)
                logo.style_xticks(rotation=90, fmt='%d', anchor=0)

                # style using Axes methods
                logo.ax.set_ylabel("$-\Delta \Delta G$ (kcal/mol)", labelpad=-1)
                logo.ax.xaxis.set_ticks_position('none')
                logo.ax.xaxis.set_tick_params(pad=-1)
                #logo.ax.set_ylim([-6, 4])

                fig.suptitle(f"{TF} {mstat} bindingmode {str(j + 1)}")

                fig.savefig(results_path + f'{TF}_bm{binding_mode}_{mstat}_bindingmode_{str(j + 1)}_logo.pdf', format='pdf')
                fig.savefig(results_path + f'{TF}_bm{binding_mode}_{mstat}_bindingmode_{str(j + 1)}_logo.png', format='png')
                plt.close()
            print(f'Done with {TF}, binding size {binding_mode}, {mstat}.')

In [9]:
# Run script with n cores to create ProBound model, careful ProBound takes 4 cores
if __name__ == "__main__":
    
    #all_TF = os.listdir(raw_eluted_p)

    
    chunks = [all_dfs[i::num_cores] for i in range(num_cores)]
    
    
    with multiprocessing.Pool(processes=num_cores) as pool:
        # Call process_files function for each chunk
        pool.map(process_files, chunks)

Create config for ProBound and set env
Create config for ProBound and set env
Running ProBound for ZNF891_FL, exp10, binding size 6, unmethylated.
Running ProBound for ZNF891_FL, exp10, binding size 6, methylated.
Done with ZNF891_FL, binding size 6, methylated.
Done with ZNF891_FL, binding size 6, unmethylated.
